In [1]:
import numpy as np
import matplotlib.pyplot as plt
import operator
from collections import defaultdict

def entropia(p):
  if(type(p) is dict):
    pr=np.array(list(p.values()))
    pr=pr/np.sum(pr)
  else:
    pr=np.array(p)
    pr=pr/pr.sum()
  #print(pr)
  return -np.sum([pi*np.log2(pi) if pi>0 else 0 for pi in pr])


In [2]:
with open('./libros/libro_muertos.txt', encoding='latin1') as file:
  lines1 = file.readlines()
with open('./libros/constitucion.txt', encoding="utf-8") as file:
  lines2 = file.readlines()
with open('./libros/libro1.txt', encoding="utf-8") as file:
  lines3 = file.readlines()
with open('./libros/novela1.txt', encoding="utf-8") as file:
  lines4 = file.readlines()
with open('./libros/principito.txt', encoding="utf-8") as file:
  lines5 = file.readlines()
with open('./libros/quijote.txt', encoding="utf-8") as file:
  lines6 = file.readlines()

lines=lines1+lines2+lines3+lines4+lines5+lines6

In [3]:
#Calcular la entropía de las palabras
cadena = ''

for linea in lines:
    cadena = cadena +' '+ linea.lower()
#Para quitar los caracteres no alfanuméricos
permited='áéíóúñabcdefghijklmnopqrstuvwxyz '
cadena = ''.join([e for e in cadena.lower() if e in permited])
cadena=cadena.replace('á','a')
cadena=cadena.replace('é','e')
cadena=cadena.replace('í','i')
cadena=cadena.replace('ó','o')
cadena=cadena.replace('ú','u')

#Voy a retirar las tildes, pues no sirven en ahorcado
palabras = cadena.split()
contador_palabras = {}
for palabra in palabras:
  if (palabra not in contador_palabras.keys()):
    contador_palabras[palabra] = 1
  else:
    contador_palabras[palabra] += 1


In [4]:
len(palabras)

323747

In [5]:
Ho=entropia(contador_palabras)
print("H de palabras =",Ho)

H de palabras = 9.831926565190852


In [6]:
def GananciaDeInformacion(palabras, letras):
  Ho=entropia(palabras)
  contiene={}
  N=np.sum(list(palabras.values()))
  nocontiene={}
  for k,v in palabras.items():
    if(len(set(k).intersection(set(letras)))==0):
      nocontiene[k]=v
    else:
      contiene[k]=v
  prcontiene=np.sum(list(contiene.values()))/N
  prnocontiene=np.sum(list(nocontiene.values()))/N

  return Ho-(entropia(contiene)*prcontiene+entropia(nocontiene)*prnocontiene)
#GananciaDeInformacion(contador_palabras, ['a'])

def patron_resultante(patron_actual, palabra, letra):
    """
    Devuelve el patrón que aparecería si la palabra verdadera fuera 'palabra'
    y se preguntara por 'letra'.
    """
    cad='' #Cadena de retorno del análisis
    nuevo = list(patron_actual)

    for i, c in enumerate(palabra):
        if c == letra:
            nuevo[i] = letra

    return "".join(nuevo)


def ganancia_informacion(
        palabras,
        patron_actual,
        letra,
        debug=False):

    """
    palabras : diccionario
        {"casa": frecuencia, ...}

    patron_actual :
        por ejemplo "__a_"

    letra :
        letra ensayada

    retorna:
        (ganancia, H_original, H_nueva)
    """
    cad=''
    # -----------------------------
    # Entropía del conjunto actual
    # -----------------------------

    H_original = entropia(palabras)

    peso_total = sum(palabras.values())

    # -----------------------------
    # Construcción de los grupos
    # -----------------------------

    grupos = defaultdict(dict)

    for palabra, peso in palabras.items():

        p = patron_resultante(
            patron_actual,
            palabra,
            letra)

        grupos[p][palabra] = peso

    # -----------------------------
    # Entropía esperada
    # -----------------------------

    H_nueva = 0

    if debug:
        cad+="=" * 70+'\n'
        cad+=f"Patrón actual : {patron_actual}\n"
        cad+=f"Letra ensayada: '{letra}'\n"
        cad+='\n'

    for patron, grupo in sorted(grupos.items()):

        peso_grupo = sum(grupo.values())

        prob = peso_grupo / peso_total

        H_grupo = entropia(grupo)

        H_nueva += prob * H_grupo

        if debug:

            ejemplos = list(grupo.keys())[:3]

            cad+=f"Patrón: {patron}. , ejemplos : {ejemplos}. NumPalabras : {len(grupo)}. Probabilidad : {prob:.4f}. Entropía     : {H_grupo:.4f}\n"


    ganancia = H_original - H_nueva

    if debug:

        cad+="-" * 70+'\n'
        cad+=f"Entropía original : {H_original:.4f}\n"
        cad+=f"Entropía esperada : {H_nueva:.4f}\n"
        cad+=f"Ganancia          : {ganancia:.4f}\n"


    return ganancia, H_original, H_nueva, cad

def VerificarPatron(cad, patron, letras_a_quitar):
  if(len(cad)!=len(patron)):
    return False
  if(len(set(cad).intersection(letras_a_quitar))>0): #La cadena tiene una letra que no debe tener
    return False
  #print('Igual longitud, verificaremos el resto')
  igual=True
  for xi,yi in zip(cad, patron):
    if(yi!='_'):
      #print(xi)
      if(xi!=yi):
        igual=False
    else: #Si la yi es _, la xi no puede estar dentro de las usadas.
      if(xi in patron):
        igual=False
      #print('se omite')
  return igual

def PalabrasConPatron(palabras, patron, letras_usadas): #El patrón debe ser de la forma: '___a__b__c_'
  salida={}
  #Letras que deben ser quitadas, porque ya se usaron y no están en el patrón.
  letras_a_quitar=list(set(letras_usadas)-set(patron))
  for k,v in palabras.items():
    if(VerificarPatron(k,patron,letras_a_quitar)):
      salida[k]=v
  return salida
#PalabrasConPatron(contador_palabras, '_o___o_',['p','s','c'])


In [14]:
#Proceso, comenzamos con el set de palabras completo
#Definimos entonces el mejor caracter para reemplazar
palabras=contador_palabras.copy()
letras_usadas=[]
letras='abcdefghijklmnopqrstuvwxyz'
patron='_'
letra=''
debug=False
while('_' in patron and len(palabras)>1):
  patron_ant=patron
  patron=input("Introduzca el patron: ")
  palabras=PalabrasConPatron(palabras, patron, letras_usadas)
  if(len(palabras)==1):
    print('TERMINAMOS: ',list(palabras.keys()))
    break
  letras_disponibles=list(set(''.join(list(palabras.keys()))))
  letras=''.join(list(set(letras).intersection(set(letras_disponibles))))
  print('Letras disponibles',letras)
  letras
  ind=12
  sorted_palabras = dict( sorted(palabras.items(), key=operator.itemgetter(1),reverse=True))
  print('Palabras que actualmente cumplen con el patrón: ',end='')
  for k,v in sorted_palabras.items():
    if(ind>0):
      print(k,' ',end='')
    ind-=1
  mejor=-1
  letra=''
  print('')
  arrGan={}
  textos={}
  for chr in letras:
    #dI=GananciaDeInformacion(palabras, [chr])
    dI, H_original, H_nueva, cad=ganancia_informacion(palabras,patron,chr,debug=debug)
    arrGan[chr]=dI
    textos[chr]=cad
    if(dI>mejor):
      mejor=dI
      letra=chr
  print('Ganancia de información: ')
  for clave, valor in sorted(arrGan.items(),
                           key=lambda x: x[1],
                           reverse=True)[:3]:
    print('Letra: ',clave, ', Ganancia: ',valor)
    #print(textos[clave]+'\n')


  print('La mejor letra es: ',letra)
  letras=letras.replace(letra,'')
  letras_usadas=list(set(letras_usadas+[letra]))
  #debug=True

Letras disponibles pzwkjiysvrameqclnogbhudtxf
Palabras que actualmente cumplen con el patrón: respondio  caballero  republica  articulos  funciones  ejercicio  rocinante  entidades  cualquier  servicios  paragrafo  hermosura  
Ganancia de información: 
Letra:  e , Ganancia:  4.27276802891274
Letra:  a , Ganancia:  4.2215891445635805
Letra:  r , Ganancia:  3.628594834951243
La mejor letra es:  e
Letras disponibles pzjisvramqclnogbhudtf
Palabras que actualmente cumplen con el patrón: intencion  molecular  potencial  inversion  direccion  dimension  liderazgo  comenzado  invencion  aprendido  valerosos  cometidos  
Ganancia de información: 
Letra:  a , Ganancia:  3.208712819522606
Letra:  o , Ganancia:  3.090697752697268
Letra:  i , Ganancia:  2.980935862619996
La mejor letra es:  a
Letras disponibles gphbjuidfcotlsnvrm
Palabras que actualmente cumplen con el patrón: intencion  inversion  direccion  dimension  invencion  cometidos  sometidos  objetivos  divertido  ginesillo  ingenioso  co